In [1]:
# import modules
import numpy as np
import tensorflow as tf
from tensorflow import keras
import pandas as pd
import seaborn as sns
from pylab import rcParams
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupShuffleSplit
from scipy import stats
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras import optimizers
from datetime import datetime
from kerastuner.tuners import RandomSearch
import tensorflow_addons as tfa
from keras.utils import to_categorical 
from keras.regularizers import l2
from scipy.stats import mode



r_seed = 42

WORKDIR = 'C:/Users/cristiano.nascimento/git/LSTM_giroscopio_acelerometro'
np.random.seed(r_seed)
tf.random.set_seed(r_seed)
data_hora= datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
LOG_DIR = WORKDIR + "/Code/DataPrep/"+data_hora

C:\Users\cristiano.nascimento\AppData\Local\Temp\ipykernel_78204\2084090043.py:15: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner.tuners import RandomSearch
c:\Users\cristiano.nascimento\git\LSTM_giroscopio_acelerometro\.venv\lib\site-packages\tensorflow_addons\utils\tfa_eol_msg.py:23: UserWarning: 

TensorFlow Addons (TFA) has ended development and introduction of new features.
TFA has entered a minimal maintenance and release mode until a planned end of life in May 2024.
Please modify downstream libraries to take dependencies from other repositories in our TensorFlow community (e.g. Keras, Keras-CV, and Keras-NLP). 

For more information see: https://github.com/tensorflow/addons/issues/2807 

  warnings.warn(


In [2]:
data = pd.read_csv(WORKDIR+'/Data/KUHAR/3.Time_domain_subsamples/KU-HAR_time_domain_subsamples_20750x300.csv',header=None, dtype={"value": "float64"})

In [3]:
data.head()

,0,1,2,3,4,5,6,7,8,9,...,1793,1794,1795,1796,1797,1798,1799,1800,1801,1802
0,0.004230,-0.000280,-0.013906,-0.002544,0.027433,0.058300,0.051670,0.071857,0.080653,0.047917,...,-0.016692,-0.018566,-0.017361,-0.017919,-0.016680,-0.016650,-0.013097,0,300,1
1,0.012482,0.032496,0.067856,0.071552,0.078103,0.040353,-0.001059,-0.018580,-0.026813,-0.011280,...,-0.008210,-0.010408,-0.011459,-0.011747,-0.010394,-0.008070,-0.004354,0,300,2
2,0.012127,0.028458,0.062075,0.052611,0.022942,-0.010017,-0.023151,-0.028515,0.005036,0.008450,...,0.000915,0.000771,-0.002560,-0.003020,-0.004190,0.000215,0.000850,0,300,3
3,0.052964,0.074319,0.110670,0.108490,0.091825,0.056989,0.029337,-0.016724,-0.042265,-0.053983,...,-0.001209,0.002878,0.000663,0.000982,-0.002148,-0.009200,-0.010653,0,300,4
4,-0.020462,-0.030787,-0.008617,0.008906,0.045046,0.042136,0.045037,0.045182,0.025113,0.021730,...,-0.006369,-0.006614,-0.004698,-0.007279,-0.006861,-0.006161,-0.006113,0,300,5


In [4]:
data.shape

(20750, 1803)

In [5]:
data.tail()

,0,1,2,3,4,5,6,7,8,9,...,1793,1794,1795,1796,1797,1798,1799,1800,1801,1802
20745,1.8716,1.6488,1.60890,1.80230,1.47500,1.390700,1.13780,1.28960,0.96375,1.022200,...,-0.49764,-0.43259,-0.414760,-0.363110,-0.329190,-0.460960,-0.621920,9,300,20746
20746,1.8894,2.0227,0.81730,-0.50584,2.46220,3.814200,2.66100,2.65120,1.99140,4.427300,...,-0.20882,-0.28893,-0.320880,-0.379540,-0.408210,-0.507970,-0.548040,9,300,20747
20747,1.2695,1.3122,0.72557,0.53129,0.95842,-0.049679,1.22310,1.83700,0.98974,1.882600,...,1.21850,1.12540,0.639470,0.277320,0.172710,-0.002894,0.023955,9,300,20748
20748,-5.5622,-2.7970,-2.35330,-3.07560,-4.13610,-4.028800,-0.58026,0.34607,-2.63170,-2.790000,...,-0.24667,-0.20902,-0.128670,-0.104730,-0.131960,-0.150780,-0.187320,9,300,20749
20749,0.8381,-1.2116,-0.56718,-0.55532,-0.64496,-0.232160,0.36707,-0.04070,0.18598,-0.009721,...,0.26162,0.11912,0.004095,0.006755,0.013448,0.027566,0.140510,9,300,20750


In [6]:
data = data[data[1800].isin([0,1,4,5,6])]

In [7]:
data.shape

(9513, 1803)

In [8]:
data[1800].value_counts().sort_index()

1800
0    1886
1    1874
4    2178
5    1813
6    1762
Name: count, dtype: int64

In [9]:
# Define the mapping dictionary
mapping_dict = {4:2,5:3,6:4}

In [10]:
data = data.replace(to_replace=mapping_dict)


In [11]:
data[1800].value_counts().sort_index()

1800
0    1886
1    1874
2    2178
3    1813
4    1762
Name: count, dtype: int64

In [12]:
X = data.iloc[:, :-3]

In [13]:
y = data.iloc[:,-3]

In [14]:
X_train_data, X_validation_data, labels_train, labels_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# **Data Exploration**

In [15]:
X_train_data.shape

(7610, 1800)

In [16]:
labels_train.shape

(7610,)

In [17]:
X_validation_data.shape


(1903, 1800)

In [18]:
labels_val.shape

(1903,)

In [19]:
from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

def balance_data_with_smote(_x_data, label_data):
    """
    Balances the class distribution in a DataFrame using SMOTE.

    Parameters:
        data (pd.DataFrame): The input DataFrame containing features and labels.
        label_column (str): The name of the column containing class labels.

    Returns:
        pd.DataFrame: A balanced DataFrame with the same structure as the input.
    """
    # Separate features and labels
    X = _x_data
    y = label_data

    # Check class distribution before SMOTE
    print("Before SMOTE:", Counter(y))

    # Apply SMOTE
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_smote, y_smote = smote.fit_resample(X, y)

    # Check class distribution after SMOTE
    print("After SMOTE:", Counter(y_smote)) 
    print(X_smote.shape) 
    print(X_smote.shape) 

    return X_smote, y_smote

# Example usage:
# balanced_df = balance_data_with_smote(data, label_column='label')


In [20]:
X_train_data,labels_train = balance_data_with_smote(X_train_data,labels_train.values)

Before SMOTE: Counter({2: 1734, 1: 1522, 0: 1493, 3: 1457, 4: 1404})
After SMOTE: Counter({2: 1734, 4: 1734, 1: 1734, 3: 1734, 0: 1734})
(8670, 1800)
(8670, 1800)


In [21]:
print(X_train_data.shape)
print(labels_train.shape)

(8670, 1800)
(8670,)


# **Models**

In [22]:
labels_val_categorical = to_categorical(labels_val,num_classes=5)

In [23]:
print(labels_val_categorical)

[[0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 1. 0.]
 ...
 [0. 0. 0. 1. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]]


In [24]:
print(labels_val_categorical.shape)

(1903, 5)


In [25]:
#print(occurrence_transitions_slide)

In [26]:
labels_train_categorical = to_categorical(labels_train,num_classes=5)

In [27]:
print(labels_train_categorical)

[[0. 0. 1. 0. 0.]
 [0. 0. 1. 0. 0.]
 [0. 0. 0. 0. 1.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]


In [28]:
print(labels_train_categorical.shape)

(8670, 5)


In [29]:
X_train = X_train_data
y_train = labels_train_categorical
X_validation = X_validation_data
y_validation = labels_val_categorical

In [30]:
X_train = X_train.values

In [31]:
#X_train = X_train.reshape(X_train.shape[0], 1, X_train.shape[1], 1)

In [32]:
X_train = np.expand_dims(X_train, axis=-1) 

In [33]:
X_train.shape

(8670, 1800, 1)

In [34]:
X_validation = X_validation.values

In [35]:
X_validation = np.expand_dims(X_validation, axis=-1) 

In [36]:
#X_validation = X_validation.reshape(X_validation.shape[0], 1, X_validation.shape[1], 1)

In [37]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(8670, 1800, 1)
(8670, 5)
(1903, 1800, 1)
(1903, 5)


In [38]:
def shuffle_data(variable_data,label_data):
    perm_indices = np.random.permutation(variable_data.shape[0])
    # Shuffle both arrays using the same permutation indices
    shuffled_variable_data = variable_data[perm_indices]
    shuffled_label_data = label_data[perm_indices]
    return (shuffled_variable_data,shuffled_label_data)

In [39]:
#X_train,y_train = shuffle_data(X_train,y_train)
#X_validation,y_validation = shuffle_data(X_validation,y_validation)

In [40]:
print(X_train.shape)
print(y_train.shape)
print(X_validation.shape)
print(y_validation.shape)

(8670, 1800, 1)
(8670, 5)
(1903, 1800, 1)
(1903, 5)


In [41]:
#from tensorflow.keras import regularizers

In [42]:
''' def cria_LSTM(hp):
  model = tf.keras.Sequential()  
  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))
  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation='relu'))
  model.add(tf.keras.layers.BatchNormalization())  
  model.add(tf.keras.layers.Dense(units=4, activation='softmax'))
  model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
  return model
 '''

' def cria_LSTM(hp):\n  model = tf.keras.Sequential()  \n  model.add(tf.keras.layers.LSTM(hp.Int("input_units",min_value=32,max_value=256,step=32), input_shape=(200,6), return_sequences=False))\n  model.add(tf.keras.layers.Dense(hp.Int("input_units_relu",min_value=32,max_value=256,step=32), activation=\'relu\'))\n  model.add(tf.keras.layers.BatchNormalization())  \n  model.add(tf.keras.layers.Dense(units=4, activation=\'softmax\'))\n  model.compile(loss=\'categorical_crossentropy\', optimizer=tf.keras.optimizers.Adam(hp.Float(\n                    \'learning_rate\',\n                    min_value=1e-4,\n                    max_value=1e-2,\n                    sampling=\'LOG\',\n                    default=1e-3\n                )), metrics=[\'accuracy\'])\n  return model\n '

In [43]:
""" from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape
import keras.backend as K

def lstm_attention_sensor(hp):
    inputs = Input(shape=(200,6))
    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
             recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),return_sequences=True)(inputs)
    
    # Attention mechanism
    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation='relu')(lstm_out)    
    attention = Permute([2, 1])(attention)
    attention = Activation('softmax')(attention)
    attention = Permute([2, 1])(attention)
    
    context_vector = Multiply()([lstm_out, attention])
    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    
    output = Dense(6, activation='softmax')(context_vector)
    
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(hp.Float(
                    'learning_rate',
                    min_value=1e-4,
                    max_value=1e-2,
                    sampling='LOG',
                    default=1e-3
                )), metrics=['accuracy'])
    return model
 """

' from keras.models import Model\nfrom keras.layers import Input, LSTM, Dense, Dropout, Activation, Permute, Multiply, Lambda, Reshape\nimport keras.backend as K\n\ndef lstm_attention_sensor(hp):\n    inputs = Input(shape=(200,6))\n    lstm_out = LSTM(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), kernel_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),\n             recurrent_regularizer=l2(hp.Float(\'l2_reg1\', min_value=0.001, max_value=0.1, sampling=\'log\')),return_sequences=True)(inputs)\n    \n    # Attention mechanism\n    attention = Dense(hp.Int("input_units_lstm1",min_value=32,max_value=256,step=32), activation=\'relu\')(lstm_out)    \n    attention = Permute([2, 1])(attention)\n    attention = Activation(\'softmax\')(attention)\n    attention = Permute([2, 1])(attention)\n    \n    context_vector = Multiply()([lstm_out, attention])\n    context_vector = Lambda(lambda x: K.sum(x, axis=1))(context_vector)    \n    output

In [44]:
from keras.models import Model
from keras.layers import Input, LSTM, Dense, Dropout, Attention, Conv1D, MaxPooling1D, BatchNormalization
from keras.regularizers import l2
import tensorflow as tf

def lstm_attention_sensor(hp):
    inputs = Input(shape=(1800,1))  # Input shape adjusted for accelerometer and gyroscope data
    
    # CNN Layer: Extract local features
    conv_out = Conv1D(filters=hp.Int('conv_filters', min_value=16, max_value=256, step=16), 
                      kernel_size=hp.Choice('kernel_size', [3, 5]), 
                      activation='relu')(inputs)
    conv_out = MaxPooling1D(pool_size=2)(conv_out)
    conv_out = BatchNormalization()(conv_out)  # Normalize after convolution
    
    # First LSTM layer
    lstm_out = LSTM(hp.Int("input_units_lstm1", min_value=32, max_value=256, step=32), 
                    kernel_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    recurrent_regularizer=l2(hp.Float('l2_reg1', min_value=0.001, max_value=0.1, sampling='log')),
                    return_sequences=True)(conv_out)
    
    # Second LSTM layer
    lstm_out_2 = LSTM(hp.Int("input_units_lstm2", min_value=32, max_value=256, step=32), 
                      kernel_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      recurrent_regularizer=l2(hp.Float('l2_reg2', min_value=0.001, max_value=0.1, sampling='log')),
                      return_sequences=True)(lstm_out)
    
    # Dense layer for feature extraction
    dense_out = Dense(hp.Int("dense_units", min_value=32, max_value=256, step=32), activation='relu')(lstm_out_2)
    dropout_out = Dropout(hp.Float('dropout_rate', min_value=0.1, max_value=0.5, step=0.1))(dense_out)
    
    # Attention mechanism
    attention = Attention()([dropout_out, dropout_out])
    context_vector = tf.reduce_sum(attention, axis=1)
    
    # Output layer
    output = Dense(5, activation='softmax')(context_vector)
    
    # Model definition
    model = Model(inputs=inputs, outputs=output)
    model.compile(loss='categorical_crossentropy', 
                  optimizer=tf.keras.optimizers.Adam(hp.Float('learning_rate', 
                                                              min_value=1e-4, 
                                                              max_value=1e-2, 
                                                              sampling='LOG', 
                                                              default=1e-3)),
                  metrics=['accuracy'])
    
    return model


In [45]:
tuner = RandomSearch(
    lstm_attention_sensor,
    objective = 'val_accuracy', 
    seed=42,
    max_trials = 5,
    overwrite=True,        
    executions_per_trial = 1,    
    directory = LOG_DIR,
    project_name="lstm_acel_giro_multiclass")

In [46]:
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    min_delta = 0,
    verbose=1,
    restore_best_weights=True)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    verbose=1)

tensorboard = tf.keras.callbacks.TensorBoard(LOG_DIR)

my_callbacks = [early_stopping,reduce_lr,tensorboard]  


In [47]:
tuner.search(x=X_train,
            y=y_train,
            epochs=100,
            validation_data=(X_validation,y_validation), callbacks=my_callbacks,batch_size=256, use_multiprocessing=True)


Search: Running Trial #1

Value             |Best Value So Far |Hyperparameter
176               |176               |conv_filters
3                 |3                 |kernel_size
128               |128               |input_units_lstm1
0.0034974         |0.0034974         |l2_reg1
256               |256               |input_units_lstm2
0.0050547         |0.0050547         |l2_reg2
160               |160               |dense_units
0.1               |0.1               |dropout_rate
0.00098872        |0.00098872        |learning_rate

Epoch 1/100
 3/34 [=>............................] - ETA: 28:25 - loss: 144.9259 - accuracy: 0.3516

KeyboardInterrupt: 

In [ ]:
tuner.results_summary()

In [ ]:
tuner.get_best_hyperparameters()[0].values

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]

In [ ]:
best_model.history

In [ ]:
best_model.summary()

In [ ]:
loss, accuracy = best_model.evaluate(X_validation, y_validation)

In [ ]:
predictions = best_model.predict(X_validation)

In [ ]:
print(y_validation)

In [ ]:
print(y_validation.shape)

In [ ]:
y_validation_inverse = np.argmax(y_validation, axis=1)

In [ ]:
print(y_validation_inverse)

In [ ]:
print(len(y_validation_inverse))

In [ ]:
print(predictions)

In [ ]:
classes_x=np.argmax(predictions,axis=1)

In [ ]:
print(classes_x)

In [ ]:
print(len(classes_x))

In [ ]:
print(classes_x.shape)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm = confusion_matrix(y_validation_inverse, classes_x)

In [ ]:
disp = ConfusionMatrixDisplay(confusion_matrix=cm)

In [ ]:
disp.plot()

In [ ]:
print(classification_report(y_validation_inverse, classes_x))

In [ ]:
# Updated transformation function to include saving the result to a CSV file
def transform_dataframe_corrected_with_save(original_df, output_file_path):
    # Step 1: Remove columns 1801 and 1802
    reduced_df = original_df.drop(columns=[1801, 1802])
    
    # Step 2: Create a new DataFrame with 6 columns and a label column
    new_columns = ['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyro_X', 'Gyro_Y', 'Gyro_Z', 'Label']
    transformed_data = []  # Use a list to collect data for better performance
    
    # Step 3: Process each row in the original DataFrame
    for index, row in reduced_df.iterrows():
        class_id = row[1800]  # Get the class ID
        for i in range(0, 900, 300):  # Iterate over the 3 segments of accelerometer columns
            for j in range(0, 300, 3):  # Get chunks of 3 data points
                acc_x, acc_y, acc_z = row[i + j], row[i + j + 1], row[i + j + 2]
                gyro_x, gyro_y, gyro_z = row[900 + i + j], row[900 + i + j + 1], row[900 + i + j + 2]
                # Append the new row as a list
                transformed_data.append([acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z, class_id])
    
    # Convert the collected data into a DataFrame
    transformed_df = pd.DataFrame(transformed_data, columns=new_columns)
    
    # Save to a CSV file
    transformed_df.to_csv(output_file_path, index=False)
    print(f"Transformed data has been saved to {output_file_path}")

    return transformed_df

# Example usage (assuming you have the original_df loaded):
# transformed_df = transform_dataframe_corrected_with_save(original_df, "transformed_data.csv")


In [ ]:
transformed_df = transform_dataframe_corrected_with_save(data,"kuhar_dataset_transformed.csv")

In [ ]:
transformed_df.tail(1000)

In [ ]:
transformed_df.shape

In [ ]:
print(labels_test_categorical.shape)

In [ ]:
print(labels_test_categorical)

In [ ]:
loss_test, accuracy_test = best_model.evaluate(segments_test, labels_test_categorical)

In [ ]:
predictions_test = best_model.predict(segments_test)

In [ ]:
print(predictions_test)

In [ ]:
classes_x_test=np.argmax(predictions_test,axis=1)

In [ ]:
print(classes_x_test)

In [ ]:
labels_test_argmax = np.argmax(labels_test_categorical,axis=1)

In [ ]:
print(labels_test_argmax)

In [ ]:
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay,classification_report
cm_test = confusion_matrix(labels_test_argmax, classes_x_test)

In [ ]:
disp_test = ConfusionMatrixDisplay(confusion_matrix=cm_test)
disp_test.plot()

In [ ]:
print(classification_report(labels_test_argmax, classes_x_test))

In [ ]:
best_model.save(WORKDIR+"/Model/best_models_saved_"+data_hora)